In [1]:
import sys, os
import pandas as pd
import numpy as np
import itertools
from collections import defaultdict

# Notebook uses functions from .py scripts in scripts/ folder
sys.path.append('../../scripts/coordinates_analysis')

from load import load_samples
from compare import compare_samples
from save import save_overlaps

# Notebook uses functions from .py scripts in scripts/ folder
sys.path.append('../../scripts/signals_analysis')

from calculate import get_signals

# Mapping replication origins via MCM subunits binding sites


In [14]:
CHROMS = ['chrI','chrII','chrIII','chrIV','chrV','chrVI','chrVII','chrVIII','chrIX','chrX','chrXI','chrXII','chrXIII','chrXIV','chrXV','chrXVI']

In [15]:
####### LOAD DATA #######
# Load coomon peaks for MCM2 and MCM4
output_dir = '../../results/compared_peaks'
bed_path = f'{output_dir}/RDY368_G1_nTr_nT_nC_RDY412_G1_nTr_nT_nC/in_RDY412_G1_nTr_nT_nC_RDY368_G1_nTr_nT_nC_ex.bed' # output from notebooks/compare_coordinates/1_compare_peaks.ipynb
MCM_G1 = pd.read_csv(bed_path, 
                names=['chr', 'start', 'end', 'center'],
                sep='\t', header=None)

# Save as MCM_G1 in resources/
MCM_G1.to_csv(os.path.join('../../resources', 'MCM_G1.bed'), sep='\t', index=False, header=False)

In [16]:
MCM_table = pd.DataFrame()

In [17]:
# Create id for each origin
chr_list = MCM_G1['chr'].to_list()
id_list = []
d = {'chrI':0,'chrII':0,'chrIII':0,'chrIV':0,'chrV':0,'chrVI':0,'chrVII':0,'chrVIII':0,
          'chrIX':0,'chrX':0,'chrXI':0,'chrXII':0,'chrXIII':0,'chrXIV':0,'chrXV':0,'chrXVI':0}
for chromosome in chr_list:
    d[chromosome] += 1
    id_list.append(f'{chromosome}:{d[chromosome]}')
    
MCM_table['id'] = id_list
MCM_table['chr'] = MCM_G1['chr']
MCM_table['start'] = MCM_G1['start']
MCM_table['end'] = MCM_G1['end']
MCM_table['center'] = MCM_G1['center']

## Get area of peaks

In [18]:
Mcm2_G1_wig = "../../../ChEC_Data_Complete/wig/cpm_mean/RDY412_G1_nTr_nT_nC_cpm_mean.wig"
Mcm4_G1_wig = "../../../ChEC_Data_Complete/wig/cpm_mean/RDY368_G1_nTr_nT_nC_cpm_mean.wig"

Mcm2_signals = get_signals(bed_path, Mcm2_G1_wig)
Mcm4_signals = get_signals(bed_path, Mcm4_G1_wig)

In [19]:
MCM_table['Mcm2_G1_cpm_area'] = Mcm2_signals.sum(axis=0)
MCM_table['Mcm4_G1_cpm_area'] = Mcm4_signals.sum(axis=0)

In [20]:
MCM_table

,id,chr,start,end,center,Mcm2_G1_cpm_area,Mcm4_G1_cpm_area
0,chrI:1,chrI,6399,6549,6474,21140.08250,25979.94350
1,chrI:2,chrI,8119,8269,8194,29782.72620,36479.66710
2,chrI:3,chrI,10063,10213,10138,37737.97100,34678.62700
3,chrI:4,chrI,16805,16955,16880,25590.70500,29580.44540
4,chrI:5,chrI,23233,23383,23308,4095.59120,4481.11346
...,...,...,...,...,...,...,...
767,chrXVI:66,chrXVI,881198,881348,881273,48012.91100,46367.49900
768,chrXVI:67,chrXVI,909547,909697,909622,4190.87259,3315.63899
769,chrXVI:68,chrXVI,929661,929811,929736,36796.38000,32087.01070
770,chrXVI:69,chrXVI,933108,933258,933183,96044.52000,82487.77600


## Annotate mapped origins

In [21]:
ARS_SGD = pd.read_csv('../../resources/ARS_SGD.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
ARS_early = pd.read_csv('../../resources/ARS_early.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
ARS_late = pd.read_csv('../../resources/ARS_late.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
Sld3_G1 = pd.read_csv('../../../ChEC_Data_Complete/homer_peaks_merged/RDY342_G1_nTr_nT_nC.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)

## Find strongest Sld3 G1 peak in distance +-1200 bp from MCM interval

Notes:
1) Search region: Use extended area of search, by 1200bp on both sides of MCM intervals (total region = 1200 + 151 + 1200 = 2551bp)

2) Peak selection: Choose the Sld3 peak with highest height (max signal value at any position), not highest sum of signals. High peaks are steep and their tails decreasing their average, so here I'm using only highest value for comparison.

In [22]:
def get_max_signal(chr, start, end, wig_path):
    """Get sum of signals for a given region from wig file"""
    # Initialize signal storage
    signals = {pos: 0 for pos in range(start, end + 1)}
    
    chunk_size = 100000
    current_chrom = None
    
    try:
        for chunk in pd.read_csv(wig_path, chunksize=chunk_size, sep=' ', header=0):
            for row in chunk.itertuples(index=False):
                # Parse variableStep line
                if row[0] == "variableStep":
                    current_chrom = row[1].split("=")[1]
                    continue
                
                # Skip if no chromosome set
                if current_chrom is None:
                    continue
                # Skip if wrong chromosome
                if current_chrom != chr:
                    continue
                
                # Parse position and signal value
                pos, sig = int(row[0]), float(row[1])
                
                if start <= pos <= end:
                    signals[pos] = sig
        
        return max(signals.values())
    
    except FileNotFoundError:
        print(f"Error: Wig file not found at {wig_path}")
        return 0
    except Exception as e:
        print(f"Error processing wig file: {e}")
        return 0

In [23]:
def compare_pairwise(origins_dict, factor_dict, factor_wig_path, include_neighbours=0):
    ''' Input as dictionary {'MCM': MCM_df}, {'factor_name': factor_df} '''
    
    # Extract dataframes
    origins_name = list(origins_dict.keys())[0]
    origin_df = origins_dict[origins_name].copy()  # Use copy to avoid modifying original
    
    factor_name = list(factor_dict.keys())[0]
    factor_df = factor_dict[factor_name].copy()  # Use copy to avoid modifying original
    
    # Ensure factor_df has 'chr' column as string
    if 'chr' in factor_df.columns:
        factor_df['chr'] = factor_df['chr'].astype(str)
    
    # Create tracker with same index row as origin_df
    overlap_tracker = pd.DataFrame(index=origin_df.index, 
                                   columns=[f'{factor_name}_start',
                                            f'{factor_name}_end',
                                            f'{factor_name}_center'])
    
    for i, origin_row in origin_df.iterrows():
        # Find overlapping factors
        curr_overlap = factor_df[
            ((factor_df['end'] + include_neighbours) >= origin_row['start']) & 
            ((factor_df['start'] - include_neighbours) <= origin_row['end']) & 
            (factor_df['chr'].astype(str) == str(origin_row['chr']))  # Ensure same type
        ]
        
        # If no overlap for current row
        if curr_overlap.empty:
            overlap_tracker.loc[i, f'{factor_name}_start'] = np.nan
            overlap_tracker.loc[i, f'{factor_name}_end'] = np.nan
            overlap_tracker.loc[i, f'{factor_name}_center'] = np.nan
            
        elif len(curr_overlap) == 1:
            # Only one interval overlapping
            overlap_row = curr_overlap.iloc[0]
            overlap_tracker.loc[i, f'{factor_name}_start'] = overlap_row['start']
            overlap_tracker.loc[i, f'{factor_name}_end'] = overlap_row['end']
            overlap_tracker.loc[i, f'{factor_name}_center'] = overlap_row['center']
            
        else:
            # Multiple overlaps - choose one with highest signal
            peaks_signals = []
            
            # Get chromosome for this origin
            chr_val = str(origin_row['chr'])
            
            for idx, peak in curr_overlap.iterrows():
                peak_start = peak['start']
                peak_end = peak['end']
                
                # Get signal for this peak region
                signal = get_max_signal(chr_val, peak_start, peak_end, factor_wig_path)
                peaks_signals.append(signal)
            
            # Find peak with maximum signal
            max_signal_index = peaks_signals.index(max(peaks_signals))
            overlap_row = curr_overlap.iloc[max_signal_index]
            
            print(f'MCM id_{i} overlapping with {len(curr_overlap)} factors with corresponding signals {peaks_signals}. '
                  f'Chosen peak {overlap_row["start"]}-{overlap_row["end"]} with max signal {max(peaks_signals)}')
            
            overlap_tracker.loc[i, f'{factor_name}_start'] = overlap_row['start']
            overlap_tracker.loc[i, f'{factor_name}_end'] = overlap_row['end']
            overlap_tracker.loc[i, f'{factor_name}_center'] = overlap_row['center']
    
    # Combine results
    result_df = pd.concat([origin_df, overlap_tracker], axis=1)
    
    return result_df

In [24]:
MCM_table = compare_pairwise(
                            {'MCM': MCM_G1},
                            {'Sld3_G1': Sld3_G1},
                            include_neighbours=1200,
                            factor_wig_path="../../../ChEC_Data_Complete/wig/cpm_mean/RDY342_G1_nTr_nT_nC_cpm_mean.wig")

MCM_table

MCM id_15 overlapping with 2 factors with corresponding signals [466.93, 634.645]. Chosen peak 159943-160093 with max signal 634.645
MCM id_16 overlapping with 2 factors with corresponding signals [355.103, 480.483]. Chosen peak 176117-176267 with max signal 480.483
MCM id_26 overlapping with 2 factors with corresponding signals [61.7512, 81.1954]. Chosen peak 63455-63605 with max signal 81.1954
MCM id_36 overlapping with 3 factors with corresponding signals [1120.79, 481.153, 319.452]. Chosen peak 254474-254624 with max signal 1120.79
MCM id_45 overlapping with 3 factors with corresponding signals [74.3266, 78.8492, 80.1795]. Chosen peak 408325-408475 with max signal 80.1795
MCM id_76 overlapping with 3 factors with corresponding signals [43.5044, 45.6598, 126.901]. Chosen peak 31214-31364 with max signal 126.901
MCM id_77 overlapping with 3 factors with corresponding signals [539.775, 522.296, 295.842]. Chosen peak 39045-39195 with max signal 539.775
MCM id_78 overlapping with 4 fact

,chr,start,end,center,Sld3_G1_start,Sld3_G1_end,Sld3_G1_center
0,chrI,6399,6549,6474,NaN,NaN,NaN
1,chrI,8119,8269,8194,NaN,NaN,NaN
2,chrI,10063,10213,10138,NaN,NaN,NaN
3,chrI,16805,16955,16880,NaN,NaN,NaN
4,chrI,23233,23383,23308,NaN,NaN,NaN
...,...,...,...,...,...,...,...
767,chrXVI,881198,881348,881273,NaN,NaN,NaN
768,chrXVI,909547,909697,909622,NaN,NaN,NaN
769,chrXVI,929661,929811,929736,NaN,NaN,NaN
770,chrXVI,933108,933258,933183,NaN,NaN,NaN


## Check overlapping with ARS

In [25]:
def check_match(row, factor):
    overlap = factor[
        (factor['chr'] == row['chr']) & 
        (factor['start'] <= row['end'] + 150) & 
        (factor['end'] >= row['start'] - 150)
    ]
    return 1 if not overlap.empty else 0

In [26]:
# 3. Label origin based on overlapping with provided factor
MCM_table['ARS_SGD'] = MCM_table.apply(check_match, axis=1, factor=ARS_SGD).astype(int)
MCM_table['ARS_early'] = MCM_table.apply(check_match, axis=1, factor=ARS_early).astype(int)
MCM_table['ARS_late'] = MCM_table.apply(check_match, axis=1, factor=ARS_late).astype(int)
MCM_table

,chr,start,end,center,Sld3_G1_start,Sld3_G1_end,Sld3_G1_center,ARS_SGD,ARS_early,ARS_late
0,chrI,6399,6549,6474,NaN,NaN,NaN,0,0,0
1,chrI,8119,8269,8194,NaN,NaN,NaN,1,0,0
2,chrI,10063,10213,10138,NaN,NaN,NaN,0,0,0
3,chrI,16805,16955,16880,NaN,NaN,NaN,0,0,0
4,chrI,23233,23383,23308,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...
767,chrXVI,881198,881348,881273,NaN,NaN,NaN,1,0,1
768,chrXVI,909547,909697,909622,NaN,NaN,NaN,0,0,0
769,chrXVI,929661,929811,929736,NaN,NaN,NaN,0,0,0
770,chrXVI,933108,933258,933183,NaN,NaN,NaN,1,0,1


In [27]:
# Save table
MCM_table.to_csv('../../resources/MCM_table.tsv', sep='\t')

In [34]:
# Summary
print(f'There are {MCM_table["Sld3_G1_center"].notna().sum()} Sld3 G1 peaks overlapping with MCM (-+1200bp from both sides of MCM intervals)')
print(f'There are {MCM_table["ARS_SGD"].sum()} ARS_SGD peaks overlapping with MCM (-+150bp from both sides of MCM intervals)')
print(f'There are {MCM_table["ARS_early"].sum()} early ARS peaks overlapping with MCM (-+150bp from both sides of MCM intervals)')
print(f'There are {MCM_table["ARS_late"].sum()} late ARS peaks overlapping with MCM (-+150bp from both sides of MCM intervals)')

There are 161 Sld3 G1 peaks overlapping with MCM (-+1200bp from both sides of MCM intervals)
There are 302 ARS_SGD peaks overlapping with MCM (-+150bp from both sides of MCM intervals)
There are 133 early ARS peaks overlapping with MCM (-+150bp from both sides of MCM intervals)
There are 90 late ARS peaks overlapping with MCM (-+150bp from both sides of MCM intervals)


In [41]:
# Count rows where Sld3_G1_center is not NaN AND ARS_early == 1
count = ((MCM_table['Sld3_G1_center'].notna()) & (MCM_table['ARS_early'] == 1)).sum()
print(f"Number of Sld3 G1 peaks overlapping with early ARS and MCM: {count} ({round(count/len(ARS_early), 2) * 100})% of early ARS")

Number of Sld3 G1 peaks overlapping with early ARS and MCM: 96 (69.0)% of early ARS


In [42]:
# Count rows where Sld3_G1_center is not NaN AND ARS_early == 1
count = ((MCM_table['Sld3_G1_center'].notna()) & (MCM_table['ARS_late'] == 1)).sum()
print(f"Number of Sld3 G1 peaks overlapping with late ARS and MCM: {count} ({round(count/len(ARS_late), 2) * 100})% of late ARS")

Number of Sld3 G1 peaks overlapping with late ARS and MCM: 5 (5.0)% of late ARS
